# CampusCrew — notebook edition (SQLite)
Same logic as your original MySQL notebook, now using the shared `core.py` that powers the website. No MySQL server and no `input()` prompts needed.

Run the cells top to bottom. The last cell starts the web UI right inside Jupyter.

In [ ]:
%pip install -q Flask pandas plotly

In [ ]:
import os
os.environ['DATABASE_PATH'] = 'instance/notebook_demo.db'   # separate DB so demos don't touch your real data
from core import CampusCrew
cc = CampusCrew()   # creates tables + seeds the default clubs
cc.list_clubs()
import pandas as pd
pd.DataFrame(cc.list_clubs())

## Add volunteers and an event

In [ ]:
cc.add_club('Robotics Club', 'Technical')
for n, e, s, r in [('Asha','asha@college.edu','python, web','Tech'),
                   ('Ben','ben@college.edu','design, photoshop','Art'),
                   ('Chitra','chitra@college.edu','communication','PR/Management')]:
    cc.add_volunteer(n, e, s, r)

event_id = cc.add_event('GDG', 'Hack Night', '2026-11-10 10:00', '2026-11-10 14:00', 'Main Hall',
                        tech_count=1, art_count=1, pr_count=1)
pd.DataFrame(cc.list_roles(event_id))

## Recommend and assign

In [ ]:
role_id = cc.list_roles(event_id)[0]['role_id']
role, recs = cc.recommend_volunteers(role_id)
display(pd.DataFrame(recs))
print(cc.assign_volunteer(role_id, recs[0]['volunteer_id']))

In [ ]:
pd.DataFrame(cc.list_assignments())

## Attendance and analytics

In [ ]:
cc.mark_attendance(cc.list_assignments()[0]['assignment_id'], 1, 4)
va = pd.DataFrame(cc.volunteer_analytics())
display(va)
import plotly.express as px
px.bar(va, x='name', y='total_assignments', title='Volunteer Workload').show()

In [ ]:
pd.DataFrame(cc.role_report()).to_csv('campuscrew_role_capacity_report.csv', index=False)
va.to_csv('campuscrew_volunteer_analytics.csv', index=False)

## Launch the web UI from the notebook
This runs the site at http://localhost:5000 while the notebook stays usable. (For a public link, see `DEPLOY.md`.)

In [ ]:
import threading
os.environ['DATABASE_PATH'] = 'instance/campuscrew.db'   # the real DB used by the website
import importlib, core; importlib.reload(core)
from app import app
threading.Thread(target=lambda: app.run(port=5000, use_reloader=False), daemon=True).start()
print('Open http://localhost:5000')